# Base case: feeder without storage or PV

This notebook sets a **reference** for the other examples in this folder: a distribution feeder with **loads only** (no DER).

In [1]:
!pip install py-dss-toolkit


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!git clone https://github.com/PauloRadatz/opendss-python-examples

Cloning into 'opendss-python-examples'...


In [3]:
dss_file_path = "/content/opendss-python-examples/feeder_models/EPRITestCircuits/ckt5/Master_ckt5.dss"

In [4]:
import os
import pathlib
import pandas as pd
import plotly.graph_objects as go

import py_dss_interface
from py_dss_toolkit import dss_tools

In [5]:
dss = py_dss_interface.DSS()
dss_tools.update_dss(dss)
dss.started


True

In [6]:
dss.text(f"compile [{dss_file_path}]")
dss.text(f"solve")

'You must create a new circuit object first: "new circuit.mycktname" to execute this command.'

## Peak load (snapshot)

The first solve uses the peak load condition case. The summary table’s **P feeder (kW)** is the active power the substation must **deliver** to supply this feeder. In peak periods, this import is high and the **transmission and substation** are more heavily loaded—this is the kind of condition a battery can help **relieve** in a peak-shaving study.


In [7]:
peak_load_results = dss_tools.results.summary_df
peak_load_results

,Results
P feeder (kW),-0.0
Q feeder (kvar),-0.0
P losses (kW),0.0
Q losses (kvar),0.0
max voltage (pu),0.0
min voltage (pu),0.0


In [8]:
dss_tools.interactive_view.voltage_profile(title="Voltage Profile | Peak Load Condition")

ValueError: One enerymeter should exist to plot the voltage profile.

In [ ]:
dss_tools.interactive_view.active_power_settings.colorscale = "jet"
dss_tools.interactive_view.active_power_settings.colorbar_cmax = 8000
dss_tools.interactive_view.active_power_settings.colorbar_cmin = -2000
dss_tools.interactive_view.circuit_plot("active power", title="Active Power | Peak Load Condition | No Generation")

## Off-peak load (snapshot)

The same model is re-solved with a **light-load** condition. Import drops sharply, and the **voltage and loss** pattern change with it.

Two static snapshots are not enough to schedule storage; they only bracket the range. The **24-hour load shape** later in the notebook shows **which hour** is actually the peak and how the feederhead P moves through the day.


In [ ]:
dss.text(f"compile [{dss_file_path}]")
dss.text("set loadmult=0.23863636")
dss.text("solve")

In [ ]:
off_peak_load_results = dss_tools.results.summary_df
off_peak_load_results

In [ ]:
dss_tools.interactive_view.voltage_profile(title="Voltage Profile | Off-Peak Load Condition")

In [ ]:
dss_tools.interactive_view.active_power_settings.colorscale = "jet"
dss_tools.interactive_view.active_power_settings.colorbar_cmax = 8000
dss_tools.interactive_view.active_power_settings.colorbar_cmin = -2000
dss_tools.interactive_view.circuit_plot("active power", title="Active Power | Off-Peak Load Condition | No Generation")

## Comparing the two snapshot conditions

The table below compares key summary quantities between **peak** and **off-peak**. It makes clear that the same network must be secure over a **wide** range of conditions. The following section adds a **daily** simulation so the **time alignment** of load is visible hour by hour.


In [ ]:
comparison_df = pd.concat([peak_load_results, off_peak_load_results], axis=1)
comparison_df.columns = ['Peak Load', 'Off-Peak Load']
comparison_df

## Daily (24-hour) load shape

Loads follow a **one-day** curve (repeated in OpenDSS `daily` mode with **1-hour** steps in this run). The plot at the end shows **P vs. time** at the monitor on the line at the source (feederhead), so you can see **when** the feeder head is stressed and how a time-shifting resource would interact with the day. This is the time-series context the **storage** examples build on.


In [ ]:
dss.text(f"compile [{dss_file_path}]")
dss.text("New Loadshape.load_curve npts=24 interval=1 mult=(0.39204545 0.28977273 0.25568182 0.23863636 0.3125 0.48295455 0.57954545 0.45454545 0.51136364 0.51704545 0.58522727 0.59090909 0.63068182 0.55681818 0.53409091 0.53409091 0.58522727 0.72159091 0.86363636 0.90340909 1.0 0.85795455 0.73863636 0.51136364)")
dss.text("batchedit load..* daily=load_curve")
dss_tools.model.add_line_in_vsource(add_meter=True, add_monitors=True)

In [ ]:
dss.text("set mode=daily")
dss.text("solve")

In [ ]:
dss.loadshapes.name = "load_curve"
p_mult = dss.loadshapes.p_mult
t = [i + 1 for i in range(24)]

# Create the Plotly figure
fig = go.Figure(
    data=go.Scatter(x=t, y=p_mult, mode='lines+markers'),
    layout=go.Layout(
        title=go.layout.Title(text="Load Curve for All Loads"),
        xaxis=go.layout.XAxis(title=go.layout.xaxis.Title(text="Time (hours)")),
        yaxis=go.layout.YAxis(title=go.layout.yaxis.Title(text="Active Power (pu)"))
    )
)

# Display the plot
fig.show()

In [ ]:
dss_tools.interactive_view.p_vs_time(name="monitor_feeder_head_pq", title="Active Power vs Time at Feeder Head")

## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [pauloradatz.me](https://www.pauloradatz.me)
- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)

## Contact

For questions or follow-up about these materials:

- Paulo Radatz
- Email: [paulo.radatz@gmail.com](mailto:paulo.radatz@gmail.com)
- LinkedIn: [linkedin.com/in/pauloradatz](https://www.linkedin.com/in/pauloradatz/)
- Website: [pauloradatz.me](https://www.pauloradatz.me/)